In [1]:
# asl_teaching_feedback.py
"""
ASL teaching app with per-finger comparison between reference image keypoints
and live webcam keypoints. Displays side-by-side reference + live frames,
draws MediaPipe landmarks, and gives per-finger "bend/extend" feedback.
"""
import os
import cv2
import yaml
import glob
import time
import numpy as np
from collections import deque
from ultralytics import YOLO
import mediapipe as mp
from pathlib import Path

# =============================================================================
# Config
# =============================================================================
class Config:
    DATA_YAML_PATH = r"C:\Users\kusal\OneDrive\Documents\Uni stuff\Year 5 Sem 2\FYP-B\Datasets\asl-dataset-p9yw8 v2\data.yaml"
    MODEL_PATH = "runs/detect/asl-dataset-p9yw8 v2/weights/best.pt"
    CONF_THRESHOLD = 0.7
    PAUSE_DURATION = 1.5
    MAX_SENTENCE_LENGTH = 10
    DISPLAY_SCALE = 1.2
    SEQ_LEN = 64
    REFERENCE_FILE = "reference_keypoints.npz"
    EMA_ALPHA = 0.6
    MATCH_ANGLE_THRESHOLD = 20.0   # degrees: below => OK, above => feedback
    # visual params
    REF_WIDTH = 360
    LIVE_WIDTH = 640
    PANEL_HEIGHT = 300

# =============================================================================
# Keypoint utility / MediaPipe Holistic wrapper
# =============================================================================
class HandKeypointAnalyzer:
    def __init__(self):
        self.holistic = mp.solutions.holistic.Holistic(static_image_mode=False, model_complexity=0)
        self.mp_drawing = mp.solutions.drawing_utils
        self.mp_styles = mp.solutions.drawing_styles
        self.reference_keypoints = {}          # {sign_name: {'left':arr,'right':arr,'pose':arr}}
        self.ema_alpha = Config.EMA_ALPHA
        self.last_kp = None
        self._load_references_on_disk_or_images()

    def _load_references_on_disk_or_images(self):
        # attempt to load precomputed references
        p = Path(Config.REFERENCE_FILE)
        if p.exists():
            try:
                data = np.load(p, allow_pickle=True)
                for k in data.files:
                    self.reference_keypoints[k] = data[k].tolist()
                print(f"Loaded {len(self.reference_keypoints)} reference(s) from {p}")
            except Exception as e:
                print("Could not load reference keypoints:", e)
        # note: actual mapping from class name -> image path handled at ASLTeachingApp init,
        # we will compute missing references on demand via extract_keypoints_from_image()

    def save_references_to_disk(self):
        try:
            np.savez_compressed(Config.REFERENCE_FILE, **self.reference_keypoints)
            print("Saved reference_keypoints to", Config.REFERENCE_FILE)
        except Exception as e:
            print("Failed saving references:", e)

    def extract_keypoints_from_image(self, image_path):
        """
        Run holistic on a static image and return kp_dict {'left','right','pose'} (arrays or None).
        Coordinates are normalized (x,y,z) as MediaPipe returns.
        """
        img = cv2.imread(str(image_path))
        if img is None:
            return None
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        # for static image, better to use static_image_mode True temporarily
        with mp.solutions.holistic.Holistic(static_image_mode=True, model_complexity=0) as h:
            res = h.process(img_rgb)
        out = {'left': None, 'right': None, 'pose': None}
        if getattr(res, 'pose_landmarks', None):
            out['pose'] = np.array([[lm.x, lm.y, lm.z] for lm in res.pose_landmarks.landmark], dtype=np.float32)
        if getattr(res, 'left_hand_landmarks', None):
            out['left'] = np.array([[lm.x, lm.y, lm.z] for lm in res.left_hand_landmarks.landmark], dtype=np.float32)
        if getattr(res, 'right_hand_landmarks', None):
            out['right'] = np.array([[lm.x, lm.y, lm.z] for lm in res.right_hand_landmarks.landmark], dtype=np.float32)
        return out

    def extract_keypoints(self, frame):
        """Process live frame (non-static) and smooth via EMA."""
        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        res = self.holistic.process(img_rgb)
        out = {'left': None, 'right': None, 'pose': None}
        if getattr(res, 'pose_landmarks', None):
            out['pose'] = np.array([[lm.x, lm.y, lm.z] for lm in res.pose_landmarks.landmark], dtype=np.float32)
        if getattr(res, 'left_hand_landmarks', None):
            out['left'] = np.array([[lm.x, lm.y, lm.z] for lm in res.left_hand_landmarks.landmark], dtype=np.float32)
        if getattr(res, 'right_hand_landmarks', None):
            out['right'] = np.array([[lm.x, lm.y, lm.z] for lm in res.right_hand_landmarks.landmark], dtype=np.float32)
        out = self._smooth_keypoints(out)
        return out, res

    def _smooth_keypoints(self, kp_dict):
        if self.last_kp is None:
            self.last_kp = {}
            for k in ['left','right','pose']:
                v = kp_dict.get(k)
                self.last_kp[k] = v.copy() if v is not None else None
            return kp_dict
        sm = {}
        a = self.ema_alpha
        for k in ['left','right','pose']:
            cur = kp_dict.get(k)
            last = self.last_kp.get(k)
            if cur is None:
                sm[k] = last
            else:
                if last is None:
                    sm[k] = cur
                else:
                    try:
                        sm[k] = a*cur + (1.0-a)*last
                    except Exception:
                        sm[k] = cur
            self.last_kp[k] = sm[k]
        return sm

    def draw_landmarks(self, image, res, draw_face=False):
        """Draw pose and hand landmarks on the BGR image (in place)."""
        try:
            if getattr(res, 'pose_landmarks', None):
                self.mp_drawing.draw_landmarks(
                    image, res.pose_landmarks, mp.solutions.holistic.POSE_CONNECTIONS,
                    landmark_drawing_spec=self.mp_styles.get_default_pose_landmarks_style()
                )
            if getattr(res, 'left_hand_landmarks', None):
                self.mp_drawing.draw_landmarks(
                    image, res.left_hand_landmarks, mp.solutions.holistic.HAND_CONNECTIONS,
                    landmark_drawing_spec=self.mp_styles.get_default_hand_landmarks_style()
                )
            if getattr(res, 'right_hand_landmarks', None):
                self.mp_drawing.draw_landmarks(
                    image, res.right_hand_landmarks, mp.solutions.holistic.HAND_CONNECTIONS,
                    landmark_drawing_spec=self.mp_styles.get_default_hand_landmarks_style()
                )
            if draw_face and getattr(res, 'face_landmarks', None):
                self.mp_drawing.draw_landmarks(image, res.face_landmarks, mp.solutions.holistic.FACEMESH_TESSELATION,
                                               landmark_drawing_spec=self.mp_styles.get_default_face_mesh_landmarks_style())
        except Exception as e:
            # don't crash display
            print("Landmark draw error:", e)
        return image

    # finger-angle utilities
    def calculate_finger_angles(self, hand_kp):
        """Return dict finger_name -> angle at middle joint (deg). If hand_kp is None -> None."""
        if hand_kp is None:
            return None
        kp = hand_kp
        angles = {}
        mapping = {
            'thumb': (2,3,4),
            'index': (5,6,7),
            'middle': (9,10,11),
            'ring': (13,14,15),
            'pinky': (17,18,19)
        }
        for name,(a,b,c) in mapping.items():
            p1 = kp[a][:2]; p2 = kp[b][:2]; p3 = kp[c][:2]
            v1 = p1 - p2; v2 = p3 - p2
            denom = (np.linalg.norm(v1)*np.linalg.norm(v2))
            if denom < 1e-6:
                angles[name] = 0.0
            else:
                cosv = np.clip(np.dot(v1,v2)/denom, -1.0, 1.0)
                angles[name] = float(np.degrees(np.arccos(cosv)))
        return angles

    def compare_hand_shape_verbose(self, live_kp_dict, ref_kp_dict, angle_threshold=Config.MATCH_ANGLE_THRESHOLD):
        """
        Compare live vs reference per finger. Returns dict:
          {'left'/'right': {'similarity':float, 'per_finger': {finger: {'ref_angle', 'live_angle','diff','advice'}} }, 'best_side': 'left'/'right' }
        If a side missing, it will be omitted.
        Advice = 'ok' or 'extend' or 'bend'
        """
        if live_kp_dict is None or ref_kp_dict is None:
            return None

        results = {}
        sides = ['left','right']
        for side in sides:
            live = live_kp_dict.get(side)
            ref = ref_kp_dict.get(side)
            if live is None or ref is None:
                continue
            live_angles = self.calculate_finger_angles(live)
            ref_angles = self.calculate_finger_angles(ref)
            if live_angles is None or ref_angles is None:
                continue
            per_finger = {}
            total_diff = 0.0; comps = 0
            for f in ['thumb','index','middle','ring','pinky']:
                la = live_angles.get(f, 0.0)
                ra = ref_angles.get(f, 0.0)
                diff = la - ra  # positive -> live more bent than ref (assuming angle increases with bend)
                absdiff = abs(diff)
                comps += 1
                total_diff += absdiff
                if absdiff <= angle_threshold:
                    advice = 'ok'
                else:
                    # if live angle larger (more bent) -> suggest extend; else suggest bend
                    advice = 'bend' if diff > 0 else 'extend'
                per_finger[f] = {'ref_angle': ra, 'live_angle': la, 'diff': diff, 'absdiff': absdiff, 'advice': advice}
            avg_diff = total_diff / max(1, comps)
            similarity = max(0.0, 100.0 * (1.0 - (avg_diff / 90.0)))
            results[side] = {'similarity': similarity, 'avg_absdiff': avg_diff, 'per_finger': per_finger}
        # choose best side by similarity if both present
        best_side = None
        if results:
            best_side = max(results.items(), key=lambda x: x[1]['similarity'])[0]
        return {'sides': results, 'best_side': best_side}

# =============================================================================
# Sentence builder (unchanged)
# =============================================================================
class SentenceBuilder:
    def __init__(self, class_names):
        self.class_names = class_names
        self.sign_buffer = []
        self.current_word = None
        self.last_detection_time = None
        self.detection_counts = {}
        self.min_detections = 3
        self.confirm_pause = Config.PAUSE_DURATION
    def add_detection(self, class_id, confidence):
        t = time.time()
        if self.last_detection_time and (t - self.last_detection_time > self.confirm_pause):
            self._confirm_current_word()
            self.detection_counts = {}
        self.detection_counts[class_id] = self.detection_counts.get(class_id,0) + 1
        if self.detection_counts[class_id] >= self.min_detections:
            self.current_word = (class_id, confidence)
        self.last_detection_time = t
    def _confirm_current_word(self):
        if self.current_word and len(self.sign_buffer) < Config.MAX_SENTENCE_LENGTH:
            class_id, _ = self.current_word
            word = self.class_names[class_id]
            if not self.sign_buffer or self.sign_buffer[-1] != word:
                self.sign_buffer.append(word)
            self.current_word = None
    def get_current_word(self):
        if self.current_word:
            class_id, _ = self.current_word
            return self.class_names[class_id]
        return None
    def get_sentence(self):
        return " ".join(self.sign_buffer)
    def clear_sentence(self):
        self.sign_buffer = []
        self.current_word = None
        self.detection_counts = {}
        self.last_detection_time = None
    def validate_grammar(self):
        if len(self.sign_buffer) < 2:
            return True, ""
        question_words = ['WHO','WHAT','WHERE','WHEN','WHY','HOW']
        if any(w in self.sign_buffer for w in question_words):
            if self.sign_buffer[0] not in question_words:
                return False, "ASL questions typically start with question word"
        return True, ""

# =============================================================================
# Main app
# =============================================================================
class ASLTeachingApp:
    def __init__(self):
        with open(Config.DATA_YAML_PATH, 'r') as f:
            self.data_yaml = yaml.safe_load(f)
        self.class_names = self.data_yaml['names']
        self.model = YOLO(Config.MODEL_PATH)
        self.keypoint_analyzer = HandKeypointAnalyzer()
        self.sentence_builder = SentenceBuilder(self.class_names)
        # reference list maps class_id -> image_path (one image per class)
        self.reference_list = self._load_reference_images_one_per_class()
        # ensure references' keypoints exist (compute if missing)
        self._ensure_reference_keypoints()
        self.current_reference_idx = 0
        self.mode = 'teaching'
        self.seq_buffer = deque(maxlen=Config.SEQ_LEN)

    def _load_reference_images_one_per_class(self):
        img_per_class = {}
        # check common splits 'valid','test','train'
        for split in ['valid','test','train']:
            if split not in self.data_yaml:
                continue
            images_dir = self.data_yaml[split]
            if not os.path.isabs(images_dir):
                images_dir = os.path.join(os.path.dirname(os.path.abspath(Config.DATA_YAML_PATH)), images_dir)
            label_dir = images_dir.replace('images','labels')
            files = glob.glob(os.path.join(images_dir, '*.jpg')) + glob.glob(os.path.join(images_dir, '*.png')) + glob.glob(os.path.join(images_dir, '*.jpeg'))
            for p in files:
                lab = os.path.join(label_dir, os.path.splitext(os.path.basename(p))[0] + '.txt')
                if os.path.exists(lab):
                    with open(lab,'r') as lf:
                        line = lf.readline().strip()
                        if not line: continue
                        cid = int(line.split()[0])
                        if cid not in img_per_class:
                            img_per_class[cid] = p
                            # stop early if all classes found
                            if len(img_per_class) == len(self.class_names):
                                break
            if len(img_per_class) == len(self.class_names):
                break
        # produce list of (image_path, class_name) sorted by class index
        result = []
        for c in sorted(img_per_class.keys()):
            result.append((img_per_class[c], self.class_names[c]))
        return result

    def _ensure_reference_keypoints(self):
        # For each reference image, compute and store keypoints under the class name if not already stored
        for img_path, class_name in self.reference_list:
            if class_name in self.keypoint_analyzer.reference_keypoints:
                continue
            kp = self.keypoint_analyzer.extract_keypoints_from_image(img_path)
            if kp is not None:
                # store mapping by class_name
                self.keypoint_analyzer.reference_keypoints[class_name] = kp
                print(f"Computed reference keypoints for {class_name} from {img_path}")
        # persist
        self.keypoint_analyzer.save_references_to_disk()

    def _resize_keep_aspect(self, img, width=None, height=None):
        if img is None:
            return None
        h,w = img.shape[:2]
        if width is not None:
            scale = width / float(w)
            return cv2.resize(img, (width, int(h*scale)))
        if height is not None:
            scale = height / float(h)
            return cv2.resize(img, (int(w*scale), height))
        return img

    def _draw_finger_feedback_on_frame(self, frame_bgr, comparison_result, side='right'):
        """
        Draw small colored circles near the 'middle joint' and text advice for each finger.
        Assumes comparison_result is the output of compare_hand_shape_verbose and 'best_side' exists.
        We'll compute screen coords from MediaPipe normalized coordinates in live_kp_dict.
        """
        if comparison_result is None:
            return frame_bgr
        sides = comparison_result.get('sides', {})
        if side not in sides:
            return frame_bgr
        per = sides[side]['per_finger']
        # But we need the live hand keypoints to draw positions; pass them by including them in results? 
        # To keep things simple, this function expects the live normalized keypoints to be used externally to get screen coords.
        # Instead, we'll rely on that the caller will overlay circles using known 2D coords provided below.
        return frame_bgr

    def draw_ui_panel(self, annotated_live, ref_img, comp_verbose):
        # Build the right-side panel (dark) containing textual feedback
        w = annotated_live.shape[1]
        panel_h = Config.PANEL_HEIGHT
        panel = np.zeros((panel_h, w, 3), dtype=np.uint8) + 30
        y = 25
        title = f"Mode: {'TEACHING' if self.mode=='teaching' else 'SENTENCE'}"
        cv2.putText(panel, title, (10,y), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200,200,255), 2)
        y += 30
        # show reference sign name
        if len(self.reference_list) > 0:
            _, sign_name = self.reference_list[self.current_reference_idx]
            cv2.putText(panel, f"Reference: {sign_name}", (10,y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (180,255,180), 2)
        y += 28
        # show similarity & per-finger advice
        if comp_verbose is None:
            cv2.putText(panel, "No comparison available", (10,y), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (200,200,200), 1)
        else:
            best = comp_verbose.get('best_side', None)
            sides = comp_verbose.get('sides', {})
            if best and best in sides:
                sim = sides[best]['similarity']
                cv2.putText(panel, f"Best side: {best}  Similarity: {sim:.1f}%", (10,y), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (200,255,200), 1)
                y += 22
                per = sides[best]['per_finger']
                for finger in ['thumb','index','middle','ring','pinky']:
                    info = per.get(finger)
                    if not info: continue
                    adv = info['advice']
                    col = (50,200,50) if adv=='ok' else (50,50,200) if adv=='extend' else (0,0,255)
                    txt = f"{finger.capitalize():6s}: {adv} (Δ={info['absdiff']:.1f}°)"
                    cv2.putText(panel, txt, (10,y), cv2.FONT_HERSHEY_SIMPLEX, 0.5, col, 1)
                    y += 20
            else:
                cv2.putText(panel, "No hand(s) in reference/live", (10,y), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (200,200,200), 1)
        # controls
        cv2.putText(panel, "Controls: [T] Toggle Mode  [SPACE] Next Ref  [R] Save Ref  [Q] Quit", (10, panel_h-20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200,200,200), 1)
        return panel

    def _norm_to_screen(self, norm_xy, image_w, image_h):
        # norm_xy = (x,y) in [0,1]. If None, return (-1,-1)
        if norm_xy is None:
            return -1, -1
        x = int(norm_xy[0] * image_w)
        y = int(norm_xy[1] * image_h)
        return x, y

    def run(self):
        cap = cv2.VideoCapture(0)
        print("Starting ASL teaching app. Press Q to quit.")
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            orig_h, orig_w = frame.shape[:2]
            # YOLO inference
            results = self.model(frame, imgsz=416, conf=Config.CONF_THRESHOLD, verbose=False)
            # Use model's plotted frame or original depending on alignment; prefer original for landmark overlay consistency
            live_frame = frame.copy()  # we'll overlay landmarks and boxes manually
            # extract mediapipe keypoints & results
            live_kp_dict, mp_res = self.keypoint_analyzer.extract_keypoints(frame)
            # draw MP landmarks on live_frame for visualization
            live_vis = live_frame.copy()
            live_vis = self.keypoint_analyzer.draw_landmarks(live_vis, mp_res, draw_face=False)
            # draw YOLO boxes (simplified)
            try:
                boxes = results[0].boxes
                for b in boxes:
                    # b.xyxy, b.conf, b.cls
                    xyxy = b.xyxy.cpu().numpy().astype(int)[0]
                    conf = float(b.conf.cpu().numpy()[0]) if hasattr(b.conf, 'cpu') else float(b.conf)
                    cls = int(b.cls.cpu().numpy()[0]) if hasattr(b.cls, 'cpu') else int(b.cls)
                    label = f"{self.class_names[cls]} {conf:.2f}"
                    cv2.rectangle(live_vis, (xyxy[0], xyxy[1]), (xyxy[2], xyxy[3]), (0,200,0), 2)
                    cv2.putText(live_vis, label, (xyxy[0], max(0,xyxy[1]-6)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,200,0), 2)
            except Exception:
                pass

            # Reference image and keypoints
            if len(self.reference_list) > 0:
                ref_img_path, ref_name = self.reference_list[self.current_reference_idx]
                ref_kp = self.keypoint_analyzer.reference_keypoints.get(ref_name, None)
                ref_img = cv2.imread(ref_img_path)
                if ref_img is None:
                    ref_img = np.zeros((Config.REF_WIDTH, Config.REF_WIDTH, 3), dtype=np.uint8)
                ref_img = self._resize_keep_aspect(ref_img, width=Config.REF_WIDTH)
                # Draw landmarks on reference image using static extraction (recompute a small static res)
                # For the ref HUD we recompute a static result so we can use mp drawing utils
                ref_rgb = cv2.cvtColor(ref_img, cv2.COLOR_BGR2RGB)
                # compute result for drawing
                with mp.solutions.holistic.Holistic(static_image_mode=True, model_complexity=0) as h:
                    ref_res = h.process(ref_rgb)
                ref_vis = ref_img.copy()
                # ref_vis = self.keypoint_analyzer.draw_landmarks(ref_vis, ref_res, draw_face=False)
            else:
                ref_vis = np.zeros((Config.REF_WIDTH, Config.REF_WIDTH, 3), dtype=np.uint8)
                ref_name = "N/A"
                ref_kp = None

            # If in teaching mode and class matches reference, compare shapes and produce feedback
            comp_verbose = None
            try:
                # choose best detected class (highest conf) if any
                boxes = results[0].boxes
                if len(boxes) > 0:
                    confs = boxes.conf.cpu().numpy()
                    cls_arr = boxes.cls.cpu().numpy()
                    idx = int(confs.argmax())
                    best_cls = int(cls_arr[idx])
                    best_conf = float(confs[idx])
                    # if reference exists and label matches
                    if len(self.reference_list) > 0 and self.class_names[best_cls] == ref_name:
                        comp_verbose = self.keypoint_analyzer.compare_hand_shape_verbose(live_kp_dict, ref_kp)
            except Exception:
                comp_verbose = None

            # Build composite: left=ref_vis, center=live_vis resized, right=panel
            live_resized = self._resize_keep_aspect(live_vis, width=Config.LIVE_WIDTH)
            panel = self.draw_ui_panel(live_resized, ref_vis, comp_verbose)
            # overlay per-joint colored markers onto live_resized using live_kp_dict and comp_verbose
            if comp_verbose and comp_verbose.get('best_side') and live_kp_dict:
                best_side = comp_verbose.get('best_side')
                per = comp_verbose['sides'][best_side]['per_finger']
                # pick the hand to draw from live_kp_dict
                hand_kp = live_kp_dict.get(best_side)
                if hand_kp is not None:
                    h_h, h_w = live_resized.shape[:2]
                    # draw small circles for each finger middle joint (index b)
                    map_mid = {'thumb':3, 'index':6, 'middle':10, 'ring':14, 'pinky':18}
                    for finger, mid_idx in map_mid.items():
                        norm_xy = (hand_kp[mid_idx][0], hand_kp[mid_idx][1])  # normalized
                        sx, sy = self._norm_to_screen(norm_xy, h_w, h_h)
                        info = per.get(finger)
                        if info:
                            color = (0,200,0) if info['advice']=='ok' else (0,255,255) if info['advice']=='extend' else (0,0,255)
                            cv2.circle(live_resized, (sx,sy), 10, color, -1)
                            # small text
                            adv = info['advice']
                            cv2.putText(live_resized, adv[0].upper(), (sx+10, sy-6), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1)

            # Create final horizontal arrangement
            # Left: ref_vis, Middle: live_resized, Right: panel
            # ensure heights match
            target_h = max(ref_vis.shape[0], live_resized.shape[0], panel.shape[0])
            def pad_to_h(img, h):
                ih, iw = img.shape[:2]
                if ih == h:
                    return img
                pad = np.zeros((h-ih, iw, 3), dtype=np.uint8) + 10
                return np.vstack([img, pad])
            ref_p = pad_to_h(ref_vis, target_h)
            live_p = pad_to_h(live_resized, target_h)
            panel_p = pad_to_h(panel, target_h)
            composed = np.hstack([ref_p, live_p, panel_p])

            # optional scaling for display
            if Config.DISPLAY_SCALE != 1.0:
                composed = cv2.resize(composed, (0,0), fx=Config.DISPLAY_SCALE, fy=Config.DISPLAY_SCALE)

            cv2.imshow("ASL Teaching (ref | live | feedback)", composed)
            key = cv2.waitKey(1) & 0xFF
            if key == ord('q'):
                break
            elif key == ord('t'):
                self.mode = 'sentence' if self.mode == 'teaching' else 'teaching'
            elif key == ord(' ') and len(self.reference_list) > 0:
                self.current_reference_idx = (self.current_reference_idx + 1) % len(self.reference_list)
            elif key == ord('r') and len(self.reference_list) > 0:
                # capture the current live keypoints as new reference for current sign
                _, sign_name = self.reference_list[self.current_reference_idx]
                self.keypoint_analyzer.reference_keypoints[sign_name] = live_kp_dict
                self.keypoint_analyzer.save_references_to_disk()
                print(f"Saved live keypoints as reference for {sign_name}")

        cap.release()
        cv2.destroyAllWindows()

# =============================================================================
# Run
# =============================================================================
if __name__ == "__main__":
    app = ASLTeachingApp()
    app.run()

ModuleNotFoundError: No module named 'mediapipe'